# PDF Import Functionality

This notebook demonstrates the concept of importing geometric entities from PDF files and converting them to topological representations.

## Overview

PDF files often contain vector graphics that represent architectural drawings, floor plans, and technical diagrams. The goal is to:

1. Extract path data from PDF files (lines, curves, rectangles)
2. Convert paths to topologic entities (Edges, Wires, Faces)
3. Visualize and analyze the imported geometry

**Note**: The actual PDF import functionality (`Topology.ByPDFPath`) is not yet implemented in topologic_fast. This notebook demonstrates the concept using simulated PDF-like data and shows how to work with the resulting topology.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import random
import math

## Check Version

In [ ]:
print(f"topologic_fast version: {tf.__version__}")

# NOTE: Topology.ByPDFPath is not yet implemented in topologic_fast
print("\nNote: PDF import functionality is not yet implemented in topologic_fast.")
print("This notebook demonstrates the concept using simulated data.")

## PDF Path Types

PDF files contain various path types that can be converted to topologic entities:

| PDF Path Type | Topologic Entity | Description |
|--------------|------------------|-------------|
| Line (l) | Edge | Straight line segment |
| Curve (c) | Wire (approximated) | Bezier curve |
| Rectangle (re) | Face | Rectangular region |
| Quadrilateral (qu) | Face | Four-sided polygon |

Each entity can have associated attributes:
- Edge color
- Edge width
- Face color
- Face opacity

## Simulate PDF Import

Since actual PDF parsing is not available, we'll simulate importing a floor plan PDF by creating geometric entities that represent typical architectural elements.

In [ ]:
def simulate_pdf_import():
    """
    Simulate importing geometric entities from a PDF file.
    Returns a list of topologic entities with metadata.
    """
    entities = []
    metadata = []
    
    # Simulate floor plan walls (lines)
    wall_lines = [
        # Outer walls
        ((0, 0), (10, 0)),
        ((10, 0), (10, 8)),
        ((10, 8), (0, 8)),
        ((0, 8), (0, 0)),
        # Interior walls
        ((5, 0), (5, 5)),
        ((0, 5), (5, 5)),
        ((5, 5), (5, 8)),
        # Additional partitions
        ((7, 5), (10, 5)),
    ]
    
    for i, ((x1, y1), (x2, y2)) in enumerate(wall_lines):
        v1 = tf.Vertex.ByCoordinates(x1, y1, 0)
        v2 = tf.Vertex.ByCoordinates(x2, y2, 0)
        edge = tf.Edge.ByStartVertexEndVertex(v1, v2)
        entities.append(edge)
        metadata.append({
            'type': 'line',
            'edge_color': 'black',
            'edge_width': 2.0 if i < 4 else 1.5  # Outer walls thicker
        })
    
    # Simulate rooms (rectangles/faces)
    rooms = [
        {'name': 'Living Room', 'x': 0, 'y': 5, 'w': 5, 'h': 3, 'color': 'lightgreen'},
        {'name': 'Kitchen', 'x': 5, 'y': 5, 'w': 2, 'h': 3, 'color': 'peachpuff'},
        {'name': 'Dining', 'x': 7, 'y': 5, 'w': 3, 'h': 3, 'color': 'lightyellow'},
        {'name': 'Bedroom', 'x': 0, 'y': 0, 'w': 5, 'h': 5, 'color': 'lightblue'},
        {'name': 'Bathroom', 'x': 5, 'y': 0, 'w': 5, 'h': 5, 'color': 'lavender'},
    ]
    
    for room in rooms:
        face = tf.Face.Rectangle(
            room['x'] + room['w']/2,
            room['y'] + room['h']/2,
            0,
            room['w'],
            room['h']
        )
        entities.append(face)
        metadata.append({
            'type': 'rectangle',
            'name': room['name'],
            'face_color': room['color'],
            'face_opacity': 0.5
        })
    
    # Simulate doors (short lines with different style)
    doors = [
        ((2.5, 5), (3.5, 5)),  # Living/Bedroom door
        ((5, 6), (5, 7)),      # Kitchen door
        ((7.5, 5), (8.5, 5)), # Dining door
        ((5, 2), (5, 3)),      # Bathroom door
    ]
    
    for (x1, y1), (x2, y2) in doors:
        v1 = tf.Vertex.ByCoordinates(x1, y1, 0)
        v2 = tf.Vertex.ByCoordinates(x2, y2, 0)
        edge = tf.Edge.ByStartVertexEndVertex(v1, v2)
        entities.append(edge)
        metadata.append({
            'type': 'line',
            'subtype': 'door',
            'edge_color': 'brown',
            'edge_width': 3.0
        })
    
    # Simulate furniture (quadrilaterals)
    furniture = [
        {'name': 'Sofa', 'x': 1, 'y': 6, 'w': 2, 'h': 0.8},
        {'name': 'Table', 'x': 8, 'y': 6, 'w': 1.5, 'h': 1.5},
        {'name': 'Bed', 'x': 1, 'y': 1, 'w': 2, 'h': 1.8},
        {'name': 'Desk', 'x': 3.5, 'y': 1, 'w': 1, 'h': 0.6},
    ]
    
    for furn in furniture:
        face = tf.Face.Rectangle(
            furn['x'] + furn['w']/2,
            furn['y'] + furn['h']/2,
            0,
            furn['w'],
            furn['h']
        )
        entities.append(face)
        metadata.append({
            'type': 'quadrilateral',
            'name': furn['name'],
            'face_color': 'gray',
            'face_opacity': 0.7
        })
    
    return entities, metadata

# Simulate PDF import
topologic_entities, entity_metadata = simulate_pdf_import()

print(f"Simulated PDF Import:")
print(f"  Total entities: {len(topologic_entities)}")

# Count by type
type_counts = {}
for meta in entity_metadata:
    t = meta['type']
    type_counts[t] = type_counts.get(t, 0) + 1

print(f"\nBy type:")
for t, count in type_counts.items():
    print(f"    {t}: {count}")

## Visualize Imported Geometry

Let's visualize the imported entities using Plotly, similar to how `Topology.Show()` works in topologicpy.

In [ ]:
def visualize_pdf_entities(entities, metadata):
    """
    Visualize imported PDF entities in 2D.
    """
    fig = go.Figure()
    
    for entity, meta in zip(entities, metadata):
        if meta['type'] == 'line':
            # Edge/Line
            vertices = entity.Vertices()
            coords = [v.Coordinates() for v in vertices]
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            
            fig.add_trace(go.Scatter(
                x=x, y=y,
                mode='lines',
                line=dict(
                    color=meta.get('edge_color', 'black'),
                    width=meta.get('edge_width', 1.0)
                ),
                name=meta.get('subtype', 'wall'),
                showlegend=False,
                hoverinfo='text',
                hovertext=meta.get('subtype', 'wall')
            ))
        
        elif meta['type'] in ['rectangle', 'quadrilateral']:
            # Face
            vertices = entity.Vertices()
            coords = [v.Coordinates() for v in vertices]
            x = [c[0] for c in coords] + [coords[0][0]]
            y = [c[1] for c in coords] + [coords[0][1]]
            
            fig.add_trace(go.Scatter(
                x=x, y=y,
                fill='toself',
                fillcolor=meta.get('face_color', 'lightgray'),
                opacity=meta.get('face_opacity', 0.5),
                line=dict(color='black', width=1),
                name=meta.get('name', 'shape'),
                hoverinfo='text',
                hovertext=meta.get('name', 'shape')
            ))
    
    fig.update_layout(
        title='Imported PDF Floor Plan',
        xaxis=dict(title='X (units)', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y (units)'),
        width=800,
        height=700,
        showlegend=True
    )
    
    return fig

fig = visualize_pdf_entities(topologic_entities, entity_metadata)
fig.show()

## Filter by Entity Type

Just like the topologicpy PDF import, we can filter which types of entities to include or exclude.

In [ ]:
def filter_entities(entities, metadata, include_types=None, exclude_types=None):
    """
    Filter entities by type.
    
    Parameters
    ----------
    entities : list
        List of topologic entities
    metadata : list
        List of metadata dictionaries
    include_types : list, optional
        Types to include (e.g., ['line', 'rectangle'])
    exclude_types : list, optional
        Types to exclude (e.g., ['quadrilateral'])
    
    Returns
    -------
    tuple
        Filtered (entities, metadata)
    """
    filtered_entities = []
    filtered_metadata = []
    
    for entity, meta in zip(entities, metadata):
        entity_type = meta['type']
        
        if include_types and entity_type not in include_types:
            continue
        if exclude_types and entity_type in exclude_types:
            continue
        
        filtered_entities.append(entity)
        filtered_metadata.append(meta)
    
    return filtered_entities, filtered_metadata

# Filter to show only walls (lines) and rooms (rectangles)
walls_and_rooms, walls_rooms_meta = filter_entities(
    topologic_entities, 
    entity_metadata,
    include_types=['line', 'rectangle']
)

print(f"Filtered (walls + rooms): {len(walls_and_rooms)} entities")

# Visualize filtered entities
fig_filtered = visualize_pdf_entities(walls_and_rooms, walls_rooms_meta)
fig_filtered.update_layout(title='Filtered: Walls and Rooms Only')
fig_filtered.show()

In [ ]:
# Exclude furniture (quadrilaterals)
no_furniture, no_furniture_meta = filter_entities(
    topologic_entities,
    entity_metadata,
    exclude_types=['quadrilateral']
)

print(f"Filtered (excluding furniture): {len(no_furniture)} entities")

fig_no_furn = visualize_pdf_entities(no_furniture, no_furniture_meta)
fig_no_furn.update_layout(title='Filtered: Excluding Furniture')
fig_no_furn.show()

## Analyze Imported Geometry

In [ ]:
# Analyze the geometry
print("Geometry Analysis:")
print("=" * 50)

# Analyze lines (edges)
lines = [e for e, m in zip(topologic_entities, entity_metadata) if m['type'] == 'line']
total_line_length = sum(line.Length() for line in lines)
print(f"\nLines (Walls/Doors):")
print(f"  Count: {len(lines)}")
print(f"  Total length: {total_line_length:.2f} units")

# Analyze rectangles (rooms)
rooms = [(e, m) for e, m in zip(topologic_entities, entity_metadata) if m['type'] == 'rectangle']
total_room_area = sum(entity.Area() for entity, _ in rooms)
print(f"\nRectangles (Rooms):")
print(f"  Count: {len(rooms)}")
print(f"  Total area: {total_room_area:.2f} sq units")
for entity, meta in rooms:
    print(f"    {meta.get('name', 'Room')}: {entity.Area():.2f} sq units")

# Analyze quadrilaterals (furniture)
furniture = [(e, m) for e, m in zip(topologic_entities, entity_metadata) if m['type'] == 'quadrilateral']
total_furn_area = sum(entity.Area() for entity, _ in furniture)
print(f"\nQuadrilaterals (Furniture):")
print(f"  Count: {len(furniture)}")
print(f"  Total area: {total_furn_area:.2f} sq units")

## Convert to Wires and Faces

We can attempt to merge connected lines into wires and create faces from closed wires.

In [ ]:
def edges_to_wires(edges, tolerance=0.01):
    """
    Attempt to merge connected edges into wires.
    This is a simplified version - topologicpy has more sophisticated merging.
    """
    wires = []
    
    # Try to create a wire from a subset of edges that form a closed loop
    # For simplicity, we'll create individual wires from each edge
    for edge in edges:
        try:
            vertices = edge.Vertices()
            wire = tf.Wire.ByVertices([vertices[0], vertices[1]], False)
            wires.append(wire)
        except Exception as e:
            print(f"Could not create wire: {e}")
    
    return wires

# Extract wall edges
wall_edges = [e for e, m in zip(topologic_entities, entity_metadata) 
              if m['type'] == 'line' and m.get('subtype') != 'door']

print(f"Wall edges: {len(wall_edges)}")

# Create wires from edges
wires = edges_to_wires(wall_edges)
print(f"Created {len(wires)} individual wires")

## Simulated PDF Import API

Here's what the topologicpy API looks like for PDF import. This is not yet available in topologic_fast.

In [ ]:
# NOTE: Topology.ByPDFPath is not yet implemented in topologic_fast
# The following shows the topologicpy API:

# from topologicpy.Topology import Topology
#
# pdf_path = "/path/to/floor_plan.pdf"
#
# topologic_entities = Topology.ByPDFPath(
#     pdf_path,
#     wires=False,           # Just edges, don't try to make wires
#     faces=True,            # Try to make faces from closed regions
#     includeTypes=["line", "curve", "rectangle", "quadrilateral"],
#     excludeTypes=[],
#     edgeColorKey="edge_color",
#     edgeWidthKey="edge_width",
#     faceColorKey="face_color",
#     faceOpacityKey="face_opacity",
#     tolerance=0.0001,
#     silent=False
# )
#
# Topology.Show(
#     topologic_entities,
#     showVertices=False,
#     edgeWidthKey="edge_width",
#     edgeColorKey="edge_color",
#     faceColorKey="face_color",
#     faceOpacityKey="face_opacity",
#     camera=[0, 0, 10]
# )

print("PDF Import API Reference:")
print("="*60)
print("")
print("Topology.ByPDFPath(pdf_path, **options)")
print("")
print("Parameters:")
print("  pdf_path      : str  - Path to PDF file")
print("  wires         : bool - Merge edges into wires (default: False)")
print("  faces         : bool - Create faces from closed regions (default: True)")
print("  includeTypes  : list - PDF path types to include")
print("                        ['line', 'curve', 'rectangle', 'quadrilateral']")
print("  excludeTypes  : list - PDF path types to exclude")
print("  edgeColorKey  : str  - Dictionary key for edge color")
print("  edgeWidthKey  : str  - Dictionary key for edge width")
print("  faceColorKey  : str  - Dictionary key for face color")
print("  faceOpacityKey: str  - Dictionary key for face opacity")
print("  tolerance     : float - Tolerance for geometry operations")
print("  silent        : bool - Suppress warnings/errors")
print("")
print("Returns:")
print("  list - List of topologic entities (Edges, Wires, Faces)")

## Export Imported Geometry

Once imported, the geometry can be exported to various formats.

In [ ]:
# Export room faces to mesh
room_faces = [e for e, m in zip(topologic_entities, entity_metadata) if m['type'] == 'rectangle']

print("Exporting room faces to OBJ format:")
print("="*60)

for face, (_, meta) in zip(room_faces, [(e, m) for e, m in zip(topologic_entities, entity_metadata) if m['type'] == 'rectangle']):
    mesh = tf.Mesh.ByFace(face)
    obj_content = mesh.ToOBJ()
    
    room_name = meta.get('name', 'Room')
    print(f"\n{room_name}:")
    print(f"  Vertices: {mesh.NumVertices()}")
    print(f"  Triangles: {mesh.NumTriangles()}")
    print(f"  OBJ preview:")
    for line in obj_content.split('\n')[:6]:
        print(f"    {line}")
    print("    ...")

## Summary

This notebook demonstrated the concept of PDF import for architectural drawings:

1. **PDF Path Types**: Understanding line, curve, rectangle, and quadrilateral paths
2. **Entity Creation**: Converting PDF paths to topologic entities (Edge, Wire, Face)
3. **Metadata**: Preserving color, width, and opacity attributes
4. **Filtering**: Including/excluding specific path types
5. **Analysis**: Calculating lengths, areas, and counts
6. **Visualization**: Using Plotly for 2D visualization
7. **Export**: Converting to mesh formats (OBJ/STL)

### Not Yet Implemented in topologic_fast

The following topologicpy features are not yet available:

- `Topology.ByPDFPath()` - Import geometry from PDF files
- `Topology.Show()` with dictionary-based styling
- `Dictionary` integration with topology objects
- Automatic wire merging from connected edges
- Face creation from closed wire regions

### Alternative Approaches

To work with PDF files until native support is added:

1. Use `pymupdf` or `pdfminer` to extract PDF paths
2. Convert path data to topologic_fast entities manually
3. Use the approach demonstrated in this notebook